In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 17:39:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Question 1: Install Spark and PySpark

In [2]:
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


# Question 2: Yellow November 2025

In [3]:
# Retrieve data for yellow november 2025
category = "yellow"
year = 2025
month = 11

path_in = f"data/raw/{category}/{year}/{month}/"
path_out = f"data/repartitioned/{category}/{year}/{month}"

number_partitions = 4

In [4]:
df = spark.read \
    .parquet(path_in)

In [5]:
df \
    .repartition(number_partitions) \
    .write.parquet(path_out, mode="overwrite")

# Question 3: Count records

In [6]:
# How many taxi trips were there on the 15th of November?
# Consider only trips that started on the 15th of November.

In [7]:
df.registerTempTable("data")
df.columns

/home/maria.szepek/Documents/DTCDE/github/data-engineering-zoomcamp/module6-batch/.venv/lib/python3.10/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [8]:
spark.sql("""
select 
    count(1) as number_records   
from
    data
where tpep_pickup_datetime >= '2025-11-15 00:00:00' and tpep_pickup_datetime < '2025-11-16 00:00:00' 
""").show()

+--------------+
|number_records|
+--------------+
|        162604|
+--------------+



# Question 4: Longest trip

In [9]:
# What is the length of the longest trip in the dataset in hours?
spark.sql("""
select 
    tpep_pickup_datetime, 
    tpep_dropoff_datetime,
    -- timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime) as trip_duration
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0 AS trip_duration
from data
where tpep_pickup_datetime >= '2025-11-01' AND tpep_pickup_datetime < '2025-12-01'
and tpep_dropoff_datetime >= '2025-11-01' AND tpep_dropoff_datetime < '2025-12-01'
order by trip_duration desc
limit 1

""").show()

[Stage 7:==========================================>              (12 + 4) / 16]

+--------------------+---------------------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration|
+--------------------+---------------------+-------------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|    90.646667|
+--------------------+---------------------+-------------+



# Question 5: User Interface

In [10]:
# Spark's User Interface which shows the application's dashboard runs on which local port?
# on 4040

# Question 6: Least frequent pickup location zone

In [11]:
# Load the zone lookup data into a temp view in Spark:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 17:39:53--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 99.84.245.9, 99.84.245.193, 99.84.245.157, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|99.84.245.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-09 17:39:53 (28.6 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [12]:
df_taxi_zone_lookup = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")
df_taxi_zone_lookup.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [13]:
df_taxi_zone_lookup.createOrReplaceTempView("taxi_zone_lookup")
df_taxi_zone_lookup.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [14]:
# Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?
spark.sql("""
with trips_count_per_zone as (
    select 
        z.Zone as zone,
        count(1) as trips_count
    from 
        data d
    join 
        taxi_zone_lookup z on d.PULocationID = z.LocationID
    where 
        d.tpep_pickup_datetime >= '2025-11-01' 
    and d.tpep_pickup_datetime < '2025-12-01'
    and d.tpep_dropoff_datetime >= '2025-11-01' 
    and d.tpep_dropoff_datetime < '2025-12-01'
    group by 
        z.Zone
    order by 
        trips_count asc
)

select 
    *
from 
    trips_count_per_zone
where
    trips_count = (select min(trips_count) from trips_count_per_zone)
""").show(truncate=False)


+---------------------------------------------+-----------+
|zone                                         |trips_count|
+---------------------------------------------+-----------+
|Governor's Island/Ellis Island/Liberty Island|1          |
|Eltingville/Annadale/Prince's Bay            |1          |
|Arden Heights                                |1          |
+---------------------------------------------+-----------+

